In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, 
    cross_validate,
    RandomizedSearchCV
)
from xgboost import XGBClassifier


In [ ]:
df = pd.read_csv("../data/depression_data.csv")

## The Basics

In [ ]:
# Display some basic info of the df like number of rows, columns
# the dataframe head, and descriptive info of numerical columns
print(df.shape)
display(df.head())
display(df.describe().T)

A small selection of features here with only a few numerical columns. I'll be treating this as a classification problem as the "history of mental illness" that I want to predict is a yes/no column.   
The info from the kaggle site on the dataset suggested that this data was already relatively clean and deduped but worth checking for safety:

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
# Inspect the target counts
df["History of Mental Illness"].value_counts(dropna=False)

In [ ]:
# Show target graphically - predominantly "no"
sns.countplot(data=df, x="History of Mental Illness")

In [ ]:
# Check the proportions of yes / no in the target
df["History of Mental Illness"].value_counts(normalize=True)

## Numeric Columns

In [ ]:
# Select the numerical columns
numeric_cols = list(df.select_dtypes(include=np.number).columns)

In [ ]:
# Plot histograms of each of the numeric features
df[numeric_cols].hist(figsize=(15,12))

In [ ]:
# Print out table of statistical values
print(
    f"{'Feature':<18} | {'Mean':<8} | "
    f"{'Median':<8} | {'Standard Deviation':<8}"
)
for column in numeric_cols:
    print(
        f"{column:<18} | {df[column].mean():<8.2f} | "
        f"{df[column].median():<8.2f} | {df[column].std():<8.2f}"
    )

##### Visually inspect numerical feature distributions

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Age")

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Number of Children")

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Income")

Some small differences in age, very similar distributions by number of children, very different shapes when considering income. 
To me this suggests that there is some correlation with income, where higher income customers may be less likely to havea a history of mental illness.  
This is not necessarily causal (e.g. a low income job may be unlikely to result in mental illness, but those without mental illnesses may find it easier to secure a higher salary position)

## Categorical Columns

In [ ]:
# Select categorical columns
categorical_cols = list(set(df.columns)-set(numeric_cols))
categorical_cols.remove("Name")
categorical_cols.remove("History of Mental Illness")

In [ ]:
print(categorical_cols)

##### Visually inspect categorical feature distributions

In [ ]:
pd.crosstab(
    df["History of Substance Abuse"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Marital Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Education Level"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Family History of Depression"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Alcohol Consumption"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Smoking Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Physical Activity Level"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Chronic Medical Conditions"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Sleep Patterns"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Dietary Habits"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Employment Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

These distributions suggest some small correlation between a history of mental health issues and:
 * Marital status
 * Alcohol consumption   

and larger correlation between a history of mental health issues and:
 * Sleep patterns 
 * Dietary habits
 * Employment status

## Statistical Tests

In [ ]:
# We only have 3 numerical columns, but let's look at the correlation matrix
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1
)

This correlation matrix is pretty much as I would have expected it:   
 * Age and Number of Children are positively correlated, so people are more likely to have had a greater number of children as they age - this feels rational
 * Number of Children and Income are positively correlated, so people with more income are potentially better suited to support the needs of more children - this also feels rational
 * Income and Age are slightly negatively correlated, so people are likely to be lower income as they age - this was initially surprising to see

My initial expectation was that Income would increase with age, as younger people might be in education or less senior roles which I would expect to pay less. 
On refelction there are legitimate reasons for how this could be true (older people prioritising family over high career salaries, or people who have retired sitting on a lower fixed income resulting in drawing from a pension).   
Let's review the shape of the distribution to see if it fits one or both of these hypotheses. 

In [ ]:
# Group people by their age and calculate each age's mean income
age_income_df = df.groupby("Age").agg(
    mean_income=("Income", "mean")
).reset_index()

# Plot the shape of the resulting distribution
sns.lineplot(
    x=age_income_df["Age"],
    y=age_income_df["mean_income"]
)

This distribution seems to fit both hypotheses on quick inspection: people start out at lower salaries as they enter the workforce, climbing until they peak around the 30-40 range. There is a clear drop off as people enter their 40's (potentially due to family commitments or seeking a better work life balance), and another clear decrease as customers enter their 60's (presumably due to entering retirement).   
These assumptions are likely not the entire story, but are enough that my curiousity is satisfied for the purpose of this exercise. I will likely include all three of these features in the model, as they all seem to behave as expected and none correlate strongly enough to be proxies for each other. 

Before selecting the final features to be used in modelling, I want to look at the p values of each of the features. 

In [ ]:
# Calculate the p_value for each of the categorical features
for column in categorical_cols:
    table = pd.crosstab(
        df[column],
        df["History of Mental Illness"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    print(f"Column {column}, p: {p:.6f}")

The p values of each categorical feature suggest that the relationship between the target and the `History of Substance Abuse` feature is considerably more likely than the other features to occur through random chance.   
Considering the typical p value limit is around 0.05, I will likely not include this column in my model. 

Additionally, while there are no issues with the p value of the `Chronic Medical Conditions` column I have reservations against using it in the model: it's correlation to the target seems minimal, and in a real world scenario I would worry about the potential bias that may arise based on the nature of these conditions.

In [ ]:
categorical_cols.remove("Chronic Medical Conditions")

## Train Test Split

In [ ]:
print(type(numeric_cols))

In [ ]:
feature_cols = numeric_cols + categorical_cols

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols],
    df["History of Mental Illness"].apply(
        lambda x: 1 if x == "Yes" else 0
    ),
    test_size = 0.2,
    stratify=df["History of Mental Illness"].apply(
    lambda x: 1 if x == "Yes" else 0
    ),
    random_state=42
)

## Sklearn Pipeline

In [ ]:
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

scaled_preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_cols)
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_transformer, categorical_cols)
])

## Modelling

In [ ]:
dummy = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy.fit(X_train, y_train)

#dummy_pred = dummy.predict(X_test)

In [ ]:
logistic = Pipeline([
    ("preprocessor", scaled_preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic.fit(X_train, y_train)

#logistic_pred = logistic.predict(X_test)


In [ ]:
random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    ))
])

random_forest.fit(X_train, y_train)

#random_forest_pred = random_forest.predict(X_test)

In [ ]:
models = {
    "dummy": dummy,
    "logistic": logistic,
    "random_forest": random_forest
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = ["accuracy", 
           "precision",
           "recall",
           "f1",
           "roc_auc"
]

cv_results = {}

for model in tqdm(models.keys()):
    cv_results[model] = cross_validate(
        models[model],
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

In [ ]:
print(cv_results)

In [ ]:
def print_model_scores(models, cv_results):
    print(
        f"{"Model":<13} | "
        f"{"Accuracy":<8} | "
        f"{"Precision":<9} | "
        f"{"Recall":<6} | "
        f"{"F1 Score":<8} | "
        f"{"ROC AUC"}"
    )
    for model in models.keys():
        print(
            f"{model:<13} | "
            f"{cv_results[model]["test_accuracy"].mean():<8.4f} | "
            f"{cv_results[model]["test_precision"].mean():<9.4f} | "
            f"{cv_results[model]["test_recall"].mean():<6.4f} | "
            f"{cv_results[model]["test_f1"].mean():<8.4f} | "
            f"{cv_results[model]["test_roc_auc"].mean():.4f}"
        )

In [ ]:
print_model_scores(models, cv_results)

In [ ]:
xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ))
])

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__gamma": [0, 0.1, 0.5, 1, 2],
    "model__reg_alpha": [0, 0.01, 0.1, 1],
    "model__reg_lambda": [0.1, 1, 5, 10],
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=25,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    random_state=42
)

new_search = False

best_params = {
    'model__subsample': 0.7, 
    'model__reg_lambda': 10, 
    'model__reg_alpha': 1, 
    'model__n_estimators': 300, 
    'model__min_child_weight': 5, 
    'model__max_depth': 4, 
    'model__learning_rate': 0.03, 
    'model__gamma': 2, 
    'model__colsample_bytree': 0.7
}

if new_search:
    random_search.fit(X_train, y_train)
    best_params = random_search.best_params_
    print(random_search.best_params_)
    print(random_search.best_score_)

In [ ]:
xgb_optimised = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(**best_params))
])

In [ ]:
models["xgb"] = xgb_optimised

cv_results["xgb"] = cross_validate(
        models["xgb"],
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

In [ ]:
print_model_scores(models, cv_results)